# FahMai RAG Challenge: Advanced Pipeline

This notebook contains a complete flow for high-accuracy RAG:
- **Dense Retrieval** (MiniLM embeddings)
- **Sparse Retrieval** (BM25 with Thai tokenization)
- **Hybrid Retrieval** (Reciprocal Rank Fusion)
- **Reranking** (Cross-Encoder verification)
- **Super Voting** (Multi-model consensus)

In [ ]:
from google.colab import files
# file "https://www.kaggle.com/settings"
files.upload() #upload kaggle.json (Legacy API Credentials)

!mkdir -p ~/.kaggle/

# Move the uploaded kaggle.json file to the .kaggle directory
!mv kaggle.json ~/.kaggle/

# Set secure permissions for the API key file (read-only for owner)
!chmod 600 ~/.kaggle/kaggle.json

! kaggle competitions download -c super-ai-engineer-s-6-fah-mai-rag-challenge-level-1

Saving kaggle.json to kaggle.json
100% 321k/321k [00:00<00:00, 118MB/s]



In [ ]:
# Load the data from kaggle and upload here.
!unzip super-ai-engineer-s-6-fah-mai-rag-challenge-level-1.zip

Archive:  super-ai-engineer-s-6-fah-mai-rag-challenge-level-1.zip
  inflating: data/knowledge_base/policies/cancellation_policy.md  
  inflating: data/knowledge_base/policies/membership_points_policy.md  
  inflating: data/knowledge_base/policies/return_policy.md  
  inflating: data/knowledge_base/policies/shipping_policy.md  
  inflating: data/knowledge_base/policies/warranty_policy.md  
  inflating: data/knowledge_base/products/AW-MN-001_arcwave_proview_27_4k.md  
  inflating: data/knowledge_base/products/AW-SK-001_arcwave_soundpillar_300.md  
  inflating: data/knowledge_base/products/DN-DT-001_daonuea_tower_x10.md  
  inflating: data/knowledge_base/products/DN-DT-002_daonuea_tower_x10_max.md  
  inflating: data/knowledge_base/products/DN-DT-003_daonuea_mini_pc_m1.md  
  inflating: data/knowledge_base/products/DN-DT-004_daonuea_all_in_one_27.md  
  inflating: data/knowledge_base/products/DN-DT-005_daonuea_all_in_one_24.md  
  inflating: data/knowledge_base/products/DN-LT-001_daonuea_

In [ ]:
# === CONFIGURATION ===
N_QUESTIONS = 100
DATA_DIR = "/content/data"
KB_DIR = f"{DATA_DIR}/knowledge_base"

---
## Section 0: Setup & LLM Test

First, install dependencies and test the ThaiLLM API — **without** any retrieval. This shows why RAG is needed.

ThaiLLM website: https://playground.thaillm.or.th/chat/

In [ ]:
!pip install -q sentence-transformers pythainlp rank-bm25 requests python-dotenv
import os, csv, re, time, requests, numpy as np, pandas as pd
from pathlib import Path
from google.colab import userdata
from collections import Counter
from sentence_transformers import SentenceTransformer, CrossEncoder
from pythainlp.tokenize import word_tokenize
from rank_bm25 import BM25Okapi

THAILLM_API_KEY = userdata.get('ThaiLLM_API')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.8/19.8 MB 83.6 MB/s eta 0:00:00


In [ ]:
def ask_llm(messages, model="typhoon", max_retries=5):
    """Call ThaiLLM API with retry and rate-limit handling.

    Available models: typhoon, openthaigpt, pathumma, kbtg, thalle
    """
    # Mapping user friendly names to API endpoints if necessary
    model_map = {
        "openthaigpt": "openthaigpt",
        "pathumma": "pathumma",
        "typhoon": "typhoon",
        "kbtg": "kbtg"
    }

    api_model = model_map.get(model.lower(), model)
    url = f"http://thaillm.or.th/api/{api_model}/v1/chat/completions"
    headers = {"Content-Type": "application/json", "apikey": THAILLM_API_KEY}
    payload = {
        "model": "/model",
        "messages": messages,
        "max_tokens": 2024,
        "temperature": 0,
    }

    for attempt in range(max_retries):
        try:
            resp = requests.post(url, headers=headers, json=payload, timeout=120)

            if resp.status_code == 429:
                wait = min(2 ** attempt, 30)
                print(f"  [{model}] Rate limited, waiting {wait}s...")
                time.sleep(wait)
                continue

            resp.raise_for_status()
            return resp.json()["choices"][0]["message"]["content"].strip()

        except Exception as e:
            wait = 2 ** attempt
            print(f"  [{model}] Error: {e}, retrying in {wait}s...")
            time.sleep(wait)

    return None



def parse_answer(text):
    """Extract answer number from LLM response."""
    if text is None:
        return None
    # Remove any <think>...</think> blocks (some models do chain-of-thought)
    clean = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL).strip()
    # Look for ANSWER: X pattern
    m = re.search(r"ANSWER:\s*(\d+)", clean)
    if m:
        return int(m.group(1))
    # Fallback: first standalone number 1-10
    for d in re.findall(r"\b(\d{1,2})\b", clean):
        if 1 <= int(d) <= 10:
            return int(d)
    return None

### Test the LLM without RAG

Let's ask a FahMai-specific question *without* any context. The LLM shouldn't know the answer.

In [ ]:
# Ask without context — LLM has no idea about FahMai's products
response = ask_llm([
    {"role": "user", "content": "Watch S3 Ultra กันน้ำได้กี่ ATM ครับ?"}
])
print("LLM response (no context):", response)
print("\n→ The LLM doesn't know FahMai-specific facts. We need RAG!")

  [typhoon] Error: 504 Server Error: Gateway Timeout for url: http://thaillm.or.th/api/typhoon/v1/chat/completions, retrying in 1s...
  [typhoon] Error: 502 Server Error: Bad Gateway for url: http://thaillm.or.th/api/typhoon/v1/chat/completions, retrying in 2s...
  [typhoon] Error: 504 Server Error: Gateway Timeout for url: http://thaillm.or.th/api/typhoon/v1/chat/completions, retrying in 4s...
  [typhoon] Error: 504 Server Error: Gateway Timeout for url: http://thaillm.or.th/api/typhoon/v1/chat/completions, retrying in 8s...
  [typhoon] Error: 504 Server Error: Gateway Timeout for url: http://thaillm.or.th/api/typhoon/v1/chat/completions, retrying in 16s...
LLM response (no context): None

→ The LLM doesn't know FahMai-specific facts. We need RAG!


### Load Questions

In [ ]:
questions = []
with open(f"{DATA_DIR}/questions.csv", encoding="utf-8") as f:
    for row in csv.DictReader(f):
        choices = {str(i): row[f"choice_{i}"] for i in range(1, 11)}
        questions.append({"id": int(row["id"]), "question": row["question"], "choices": choices})

print(f"Loaded {len(questions)} questions.")

Loaded 100 questions.


---
## Section 1: Dense Retrieval (MiniLM)

**Dense retrieval** converts text into vectors (embeddings) and finds relevant chunks by cosine similarity.

The pipeline: **Load docs → Chunk → Embed → Retrieve → Generate**

### 1.1 Load Knowledge Base

In [ ]:
kb_dir = Path(KB_DIR)
documents = []
for fp in sorted(kb_dir.rglob("*.md")):
    text = fp.read_text(encoding="utf-8")
    documents.append({"path": str(fp.relative_to(kb_dir)), "text": text})

print(f"Loaded {len(documents)} documents")
print(f"  products/: {sum(1 for d in documents if 'products/' in d['path'])}")
print(f"  policies/: {sum(1 for d in documents if 'policies/' in d['path'])}")
print(f"  store_info/: {sum(1 for d in documents if 'store_info/' in d['path'])}")

# Preview one document
print(f"\n--- Sample: {documents[0]['path']} ---")
print(documents[0]["text"][:500])

Loaded 118 documents
  products/: 110
  policies/: 5
  store_info/: 3

--- Sample: policies/cancellation_policy.md ---
# นโยบายการยกเลิกคำสั่งซื้อ — ร้านฟ้าใหม่

**วันที่อัปเดต:** 1 มีนาคม 2569

---

## 1. ภาพรวมนโยบาย

ฟ้าใหม่เข้าใจว่าบางครั้งลูกค้าอาจต้องการยกเลิกคำสั่งซื้อด้วยเหตุผลต่างๆ นโยบายนี้อธิบายสิทธิ์และขั้นตอนการยกเลิกคำสั่งซื้อตามสถานะของคำสั่งซื้อในขณะนั้น ความสามารถในการยกเลิกขึ้นอยู่กับสถานะคำสั่งซื้อเป็นหลัก

---

## 2. การยกเลิกตามสถานะคำสั่งซื้อ

### 2.1 สถานะ "รอชำระเงิน" (Pending Payment)

**ยกเลิกได้ทันที**

คำสั่งซื้อที่อยู่ในสถานะรอชำระเงินสามารถยกเลิกได้ทันทีโดยไม่มีค่าใช้จ่าย ผ่านแอปพลิ


### 1.2 Chunking

LLMs have limited context windows, and long documents dilute relevance. We split each document into smaller **chunks** using a fixed-size sliding window with overlap.

In [ ]:
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 200

def make_chunks(text, size, overlap):
    """Split text into overlapping fixed-size windows."""
    if len(text) <= size:
        return [text]
    chunks = []
    start = 0
    while start < len(text):
        chunks.append(text[start : start + size])
        start += size - overlap
    return chunks

# Build all chunks
chunks = []
for doc in documents:
    for window in make_chunks(doc["text"], CHUNK_SIZE, CHUNK_OVERLAP):
        chunks.append({"text": window, "source": doc["path"]})

print(f"Created {len(chunks)} chunks from {len(documents)} documents")
print(f"\n--- Sample chunk ---")
print(f"Source: {chunks[0]['source']}")
print(chunks[0]["text"][:300])

Created 542 chunks from 118 documents

--- Sample chunk ---
Source: policies/cancellation_policy.md
# นโยบายการยกเลิกคำสั่งซื้อ — ร้านฟ้าใหม่

**วันที่อัปเดต:** 1 มีนาคม 2569

---

## 1. ภาพรวมนโยบาย

ฟ้าใหม่เข้าใจว่าบางครั้งลูกค้าอาจต้องการยกเลิกคำสั่งซื้อด้วยเหตุผลต่างๆ นโยบายนี้อธิบายสิทธิ์และขั้นตอนการยกเลิกคำสั่งซื้อตามสถานะของคำสั่งซื้อในขณะนั้น ความสามารถในการยกเลิกขึ้นอยู่กับสถานะคำสั่งซื้


### 1.3 Embedding

We use `paraphrase-multilingual-MiniLM-L12-v2` — a small (118M params), multilingual embedding model that produces 384-dimensional vectors. It supports Thai out of the box and doesn't require any special prefixes.

In [ ]:
from sentence_transformers import SentenceTransformer

embed_model = SentenceTransformer("BAAI/bge-m3")

# Embed all chunks
chunk_texts = [c["text"] for c in chunks]
chunk_embeddings = embed_model.encode(chunk_texts, batch_size=64, show_progress_bar=True, normalize_embeddings=True)

print(f"Chunk embeddings shape: {chunk_embeddings.shape}")  # (n_chunks, 384)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

Batches:   0%|          | 0/9 [00:00<?, ?it/s]

Chunk embeddings shape: (542, 1024)


### 1.4 Retrieve

Embed the question, then find the most similar chunks via dot product (= cosine similarity for normalized vectors).

In [ ]:
TOP_K = 5

def dense_retrieve(query, chunk_embs, k=TOP_K):
    """Return indices of top-k most similar chunks to the query."""
    q_emb = embed_model.encode([query], normalize_embeddings=True)
    scores = np.dot(chunk_embs, q_emb.T).flatten()  # cosine similarity (vectors are normalized)
    top_idx = np.argsort(scores)[::-1][:k]
    return top_idx, scores[top_idx]

### 1.5 Generate Answer

Send the retrieved context + question + choices to the LLM and parse the answer.

In [ ]:
# A very basic system prompt — you should improve this!
SYSTEM_PROMPT = "ตอบคำถามจากข้อมูลที่ให้มา เลือกตัวเลือกที่ถูกต้องที่สุด ตอบเป็น ANSWER: X เท่านั้น"

def build_rag_prompt(question, choices, retrieved_chunks):
    """Build the user prompt with retrieved context.

    TODO: Design your own prompt format.
    Think about: How should you present the context? The choices?
    What instructions help the LLM pick the right answer?
    """
    context = "\n\n".join(
        f"--- Chunk {i+1} ---\n{c['text']}"
        for i, c in enumerate(retrieved_chunks)
    )
    choices_text = "\n".join(f"{k}. {v}" for k, v in choices.items())
    # === CUSTOMIZE THIS PROMPT ===
    return (
        f"{context}\n\n"
        f"คำถาม: {question}\n\n"
        f"ตัวเลือก:\n{choices_text}\n\n"
        f"ตอบ ANSWER: X (X คือหมายเลขตัวเลือก 1-10)"
    )

# Demo: answer Q1
q = questions[0]
idx, _ = dense_retrieve(q["question"], chunk_embeddings)
retrieved = [chunks[i] for i in idx]

prompt = build_rag_prompt(q["question"], q["choices"], retrieved)
raw = ask_llm([
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": prompt},
])
answer = parse_answer(raw)
print(f"Q{q['id']}: {q['question']}")
print(f"LLM raw: {raw}")
print(f"Parsed answer: {answer}")

Q1: Watch S3 Ultra กันน้ำได้กี่ ATM ครับ
LLM raw: ANSWER: 5
Parsed answer: 5


In [ ]:
def compare_four_models(question_obj, models=["typhoon", "openthaigpt", "pathumma", "kbtg"]):
    """Retrieves context once and asks all 4 models the same question."""
    print(f"Question: {question_obj['question']}\n")

    # 1. Retrieve Context (using Hybrid as it's usually most robust)
    # Note: Ensure hybrid_retrieve is defined. Using dense as fallback if needed.
    try:
        idx = hybrid_retrieve(question_obj['question'], chunk_embeddings)
    except NameError:
        idx, _ = dense_retrieve(question_obj['question'], chunk_embeddings)

    retrieved_chunks = [chunks[i] for i in idx]
    prompt = build_rag_prompt(question_obj['question'], question_obj['choices'], retrieved_chunks)

    # 2. Query each model
    for model_name in models:
        print(f"Querying {model_name}...")
        raw_response = ask_llm([
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": prompt}
        ], model=model_name)

        parsed = parse_answer(raw_response)
        print(f"[{model_name.upper()}] Response: {raw_response}")
        print(f"[{model_name.upper()}] Parsed Answer: {parsed}")
        print("-" * 30)

# Test with Question 1
compare_four_models(questions[0])

Question: Watch S3 Ultra กันน้ำได้กี่ ATM ครับ

Querying typhoon...
[TYPHOON] Response: ANSWER: 5
[TYPHOON] Parsed Answer: 5
------------------------------
Querying openthaigpt...
[OPENTHAIGPT] Response: <think>
คำถามคือ Watch S3 Ultra กันน้ำได้กี่ ATM จากข้อมูลที่ให้มา ต้องหาข้อมูลเกี่ยวกับความสามารถในการกันน้ำของ Watch S3 Ultra

จาก Chunk 2 ระบุว่า "มาตรฐานกันน้ำระดับ 100 เมตร (10 ATM)" ซึ่งตรงกับตัวเลือกที่ 5 คือ 10 ATM

ตัวเลือกอื่นๆ ไม่ตรงกับข้อมูลที่ให้มา ดังนั้นตัวเลือกที่ถูกต้องคือ 5
</think>

ANSWER: 5
[OPENTHAIGPT] Parsed Answer: 5
------------------------------
Querying pathumma...
[PATHUMMA] Response: <think>
Okay, let's tackle this question. The user is asking about the water resistance of the Watch S3 Ultra. The options are various ATM ratings and other codes like IP68 or IPX8.

First, I need to look through the provided chunks to find the relevant information. Let me start with Chunk 2, which is the product details. In Chunk 2, under the "รายละเอียดสินค้า" section, it sa

### 1.6 Run All Questions (Dense)

Loop through questions, retrieve, generate, and collect predictions.

In [ ]:
def run_pipeline(questions, retrieve_fn, label="dense", models=["typhoon", "openthaigpt", "pathumma", "kbtg"], n=N_QUESTIONS):
    """Run retrieval + LLM comparison on first n questions."""
    all_predictions = {model: {} for model in models}

    for i, q in enumerate(questions[:n]):
        retrieved_chunks = retrieve_fn(q["question"])
        prompt = build_rag_prompt(q["question"], q["choices"], retrieved_chunks)

        row_results = [f"Q{q['id']:>3}"]
        for model_name in models:
            raw = ask_llm([
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": prompt},
            ], model=model_name)
            pred = parse_answer(raw)
            final_pred = pred if pred else 1
            all_predictions[model_name][q["id"]] = final_pred
            row_results.append(f"{model_name}: {final_pred}")

        print(" | ".join(row_results))
        time.sleep(0.1)

    return all_predictions

In [ ]:
# Define the dense_retrieve_chunks function, which is needed by run_pipeline
def dense_retrieve_chunks(query):
    idx, _ = dense_retrieve(query, chunk_embeddings)
    return [chunks[i] for i in idx]

# Run the dense retrieval pipeline to generate dense_preds_multi
print("Running Dense Retrieval Pipeline for all models...")
dense_preds_multi = run_pipeline(questions, dense_retrieve_chunks, label='dense', models=['typhoon', 'openthaigpt', 'pathumma', 'kbtg'], n=N_QUESTIONS)




# Convert the nested dictionary to a DataFrame
# dense_preds_multi structure: {'model_name': {qid: answer, ...}, ...}
df_multi = pd.DataFrame(dense_preds_multi)

# Ensure the index is named 'id' to match the question identifier
df_multi.index.name = 'id'

# Save to CSV
df_multi.to_csv('dense_preds_multi.csv')

print("Saved dense_preds_multi.csv with the following preview:")
display(df_multi.head())

Running Dense Retrieval Pipeline for all models...
  [kbtg] Error: 502 Server Error: Bad Gateway for url: http://thaillm.or.th/api/kbtg/v1/chat/completions, retrying in 1s...
Q  1 | typhoon: 5 | openthaigpt: 5 | pathumma: 5 | kbtg: 5
Q  2 | typhoon: 7 | openthaigpt: 7 | pathumma: 7 | kbtg: 7
Q  3 | typhoon: 2 | openthaigpt: 2 | pathumma: 2 | kbtg: 2
Q  4 | typhoon: 6 | openthaigpt: 6 | pathumma: 6 | kbtg: 6
  [pathumma] Error: 502 Server Error: Bad Gateway for url: http://thaillm.or.th/api/pathumma/v1/chat/completions, retrying in 1s...
Q  5 | typhoon: 6 | openthaigpt: 6 | pathumma: 6 | kbtg: 6
Q  6 | typhoon: 8 | openthaigpt: 8 | pathumma: 8 | kbtg: 8
Q  7 | typhoon: 1 | openthaigpt: 1 | pathumma: 1 | kbtg: 1
Q  8 | typhoon: 4 | openthaigpt: 4 | pathumma: 4 | kbtg: 4
  [pathumma] Error: 502 Server Error: Bad Gateway for url: http://thaillm.or.th/api/pathumma/v1/chat/completions, retrying in 1s...
  [pathumma] Error: 504 Server Error: Gateway Timeout for url: http://thaillm.or.th/api/p

,typhoon,openthaigpt,pathumma,kbtg
id,,,,
1,5,5,5,5
2,7,7,7,7
3,2,2,2,2
4,6,6,6,6
5,6,6,6,6


In [ ]:
dense_preds_multi

{'typhoon': {1: 5,
  2: 7,
  3: 2,
  4: 6,
  5: 6,
  6: 8,
  7: 1,
  8: 4,
  9: 4,
  10: 2,
  11: 4,
  12: 1,
  13: 6,
  14: 3,
  15: 1,
  16: 1,
  17: 8,
  18: 5,
  19: 2,
  20: 2,
  21: 3,
  22: 6,
  23: 3,
  24: 3,
  25: 2,
  26: 6,
  27: 2,
  28: 7,
  29: 4,
  30: 3,
  31: 1,
  32: 2,
  33: 8,
  34: 5,
  35: 3,
  36: 2,
  37: 8,
  38: 6,
  39: 4,
  40: 8,
  41: 7,
  42: 2,
  43: 4,
  44: 1,
  45: 2,
  46: 1,
  47: 2,
  48: 2,
  49: 6,
  50: 5,
  51: 7,
  52: 4,
  53: 9,
  54: 9,
  55: 9,
  56: 9,
  57: 9,
  58: 9,
  59: 9,
  60: 9,
  61: 9,
  62: 9,
  63: 9,
  64: 5,
  65: 3,
  66: 7,
  67: 6,
  68: 1,
  69: 2,
  70: 8,
  71: 4,
  72: 7,
  73: 6,
  74: 5,
  75: 6,
  76: 1,
  77: 7,
  78: 1,
  79: 7,
  80: 1,
  81: 4,
  82: 9,
  83: 3,
  84: 4,
  85: 2,
  86: 1,
  87: 1,
  88: 3,
  89: 4,
  90: 3,
  91: 2,
  92: 3,
  93: 1,
  94: 4,
  95: 1,
  96: 1,
  97: 7,
  98: 3,
  99: 4,
  100: 3},
 'openthaigpt': {1: 5,
  2: 7,
  3: 2,
  4: 6,
  5: 6,
  6: 8,
  7: 1,
  8: 4,
  9: 4,
  10: 2,


---
## Section 2: Sparse Retrieval (BM25)

**BM25** is a keyword-matching algorithm. It scores documents by how many query terms they contain, weighted by term rarity (IDF). No neural network needed.

### 2.1 Thai Tokenization

BM25 needs tokenized text. Thai has no spaces between words, so we use `pythainlp` to segment.

In [ ]:
from pythainlp.tokenize import word_tokenize

# Demo: tokenize a Thai sentence
sample = "Watch S3 Ultra กันน้ำได้กี่ ATM ครับ?"
tokens = word_tokenize(sample, engine="newmm")
print(f"Input:  {sample}")
print(f"Tokens: {tokens}")

Input:  Watch S3 Ultra กันน้ำได้กี่ ATM ครับ?
Tokens: ['Watch', ' ', 'S', '3', ' ', 'Ultra', ' ', 'กันน้ำ', 'ได้', 'กี่', ' ', 'ATM', ' ', 'ครับ', '?']


### 2.2 Build BM25 Index

In [ ]:
from rank_bm25 import BM25Okapi

# Tokenize all chunks
tokenized_chunks = [word_tokenize(c["text"], engine="newmm") for c in chunks]
bm25 = BM25Okapi(tokenized_chunks)

print(f"BM25 index built over {len(tokenized_chunks)} chunks")

BM25 index built over 542 chunks


### 2.3 Retrieve with BM25

Compare BM25 results with dense results for the same question.

In [ ]:
def bm25_retrieve(query, k=TOP_K):
    """Return top-k chunk indices using BM25."""
    tokens = word_tokenize(query, engine="newmm")
    scores = bm25.get_scores(tokens)
    top_idx = np.argsort(scores)[::-1][:k]
    return top_idx, scores[top_idx]

# Compare: same question, two retrieval methods
q = questions[0]
print(f"Question: {q['question']}\n")

d_idx, _ = dense_retrieve(q["question"], chunk_embeddings)
b_idx, _ = bm25_retrieve(q["question"])

print("Dense top-5 sources:")
for i in d_idx:
    print(f"  {chunks[i]['source']}")

print("\nBM25 top-5 sources:")
for i in b_idx:
    print(f"  {chunks[i]['source']}")

Question: Watch S3 Ultra กันน้ำได้กี่ ATM ครับ

Dense top-5 sources:
  products/WK-SW-001_wongkhojon_watch_s3_ultra.md
  products/WK-SW-001_wongkhojon_watch_s3_ultra.md
  products/WK-SW-002_wongkhojon_watch_s3_pro.md
  products/WK-SW-001_wongkhojon_watch_s3_ultra.md
  products/WK-SW-001_wongkhojon_watch_s3_ultra.md

BM25 top-5 sources:
  products/WK-SW-002_wongkhojon_watch_s3_pro.md
  products/WK-SW-001_wongkhojon_watch_s3_ultra.md
  products/WK-SW-001_wongkhojon_watch_s3_ultra.md
  products/WK-SW-003_wongkhojon_watch_s3.md
  products/WK-SW-008_wongkhojon_watch_s2_pro.md


### 2.4 Run All Questions (BM25)

In [ ]:
def bm25_retrieve_chunks(query):
    idx, _ = bm25_retrieve(query)
    return [chunks[i] for i in idx]

bm25_preds = run_pipeline(questions, bm25_retrieve_chunks, label="bm25")

Q  1 | typhoon: 5 | openthaigpt: 5 | pathumma: 5 | kbtg: 5
Q  2 | typhoon: 7 | openthaigpt: 7 | pathumma: 7 | kbtg: 7
Q  3 | typhoon: 2 | openthaigpt: 2 | pathumma: 2 | kbtg: 2
  [kbtg] Error: 502 Server Error: Bad Gateway for url: http://thaillm.or.th/api/kbtg/v1/chat/completions, retrying in 1s...
Q  4 | typhoon: 6 | openthaigpt: 6 | pathumma: 6 | kbtg: 6
Q  5 | typhoon: 6 | openthaigpt: 6 | pathumma: 6 | kbtg: 6
Q  6 | typhoon: 8 | openthaigpt: 8 | pathumma: 9 | kbtg: 8
Q  7 | typhoon: 1 | openthaigpt: 9 | pathumma: 9 | kbtg: 9
  [pathumma] Error: 502 Server Error: Bad Gateway for url: http://thaillm.or.th/api/pathumma/v1/chat/completions, retrying in 1s...
Q  8 | typhoon: 4 | openthaigpt: 4 | pathumma: 4 | kbtg: 4
  [pathumma] Error: 502 Server Error: Bad Gateway for url: http://thaillm.or.th/api/pathumma/v1/chat/completions, retrying in 1s...
  [pathumma] Error: 504 Server Error: Gateway Timeout for url: http://thaillm.or.th/api/pathumma/v1/chat/completions, retrying in 2s...
  [p

In [ ]:
# Convert the nested dictionary to a DataFrame
# dense_preds_multi structure: {'model_name': {qid: answer, ...}, ...}
df_bm25_preds = pd.DataFrame(bm25_preds)

# Ensure the index is named 'id' to match the question identifier
df_bm25_preds.index.name = 'id'

# Save to CSV
df_bm25_preds.to_csv('df_bm25_preds.csv')

print("Saved df_bm25_preds.csv with the following preview:")
display(df_bm25_preds.head())

Saved df_bm25_preds.csv with the following preview:


,typhoon,openthaigpt,pathumma,kbtg
id,,,,
1,5,5,5,5
2,7,7,7,7
3,2,2,2,2
4,6,6,6,6
5,6,6,6,6


In [ ]:
import pandas as pd

def convert_and_save_df(predictions_dict, filename_prefix):
    """
    Converts a predictions dictionary (model -> {qid -> ans}) to a DataFrame,
    sets the index name to 'id', saves it to a CSV, and displays the head.
    """
    df = pd.DataFrame(predictions_dict)
    df.index.name = 'id'
    filename = f'{filename_prefix}_preds_multi.csv'
    df.to_csv(filename)
    print(f"Saved {filename} with the following preview:")
    display(df.head())
    return df

---
## Section 3: Hybrid Retrieval (RRF)

**Hybrid** combines dense and sparse results. The idea: dense is good at semantic matching, BM25 is good at exact keyword matching. Together they cover more cases.

We use **Reciprocal Rank Fusion (RRF)** to merge the two ranked lists:

$$\text{score}(d) = \sum_{r \in \text{rankers}} \frac{1}{k + \text{rank}_r(d)}$$

where $k$ is a constant (typically 60). Documents ranked highly by *either* method get a high combined score.

In [ ]:
def hybrid_retrieve(query, chunk_embs, k=TOP_K, rrf_k=60):
    """Combine dense + BM25 results using Reciprocal Rank Fusion."""
    # Get top candidates from each method (fetch more than k to improve fusion)
    fetch_k = k * 2
    d_idx, _ = dense_retrieve(query, chunk_embs, k=fetch_k)
    b_idx, _ = bm25_retrieve(query, k=fetch_k)

    # Compute RRF scores
    rrf_scores = {}
    for rank, idx in enumerate(d_idx, 1):
        rrf_scores[idx] = rrf_scores.get(idx, 0) + 1.0 / (rrf_k + rank)
    for rank, idx in enumerate(b_idx, 1):
        rrf_scores[idx] = rrf_scores.get(idx, 0) + 1.0 / (rrf_k + rank)

    # Sort by combined score, return top-k
    sorted_idx = sorted(rrf_scores.keys(), key=lambda x: rrf_scores[x], reverse=True)[:k]
    return sorted_idx

# Demo
q = questions[0]
h_idx = hybrid_retrieve(q["question"], chunk_embeddings)
print(f"Question: {q['question']}\n")
print("Hybrid top-5 sources:")
for i in h_idx:
    print(f"  {chunks[i]['source']}")

Question: Watch S3 Ultra กันน้ำได้กี่ ATM ครับ

Hybrid top-5 sources:
  products/WK-SW-001_wongkhojon_watch_s3_ultra.md
  products/WK-SW-002_wongkhojon_watch_s3_pro.md
  products/WK-SW-001_wongkhojon_watch_s3_ultra.md
  products/WK-SW-001_wongkhojon_watch_s3_ultra.md
  products/WK-SW-001_wongkhojon_watch_s3_ultra.md


### 3.2 Run All Questions (Hybrid)

In [ ]:
def hybrid_retrieve_chunks(query):
    idx = hybrid_retrieve(query, chunk_embeddings)
    return [chunks[i] for i in idx]

# Run Hybrid retrieval for all models (100 questions)
print("Running Hybrid Retrieval Pipeline for all models...")
hybrid_preds_multi = run_pipeline(questions, hybrid_retrieve_chunks, label='hybrid', models=['typhoon', 'openthaigpt', 'pathumma', 'kbtg'], n=100)

Running Hybrid Retrieval Pipeline for all models...
  [pathumma] Error: 502 Server Error: Bad Gateway for url: http://thaillm.or.th/api/pathumma/v1/chat/completions, retrying in 1s...
Q  1 | typhoon: 5 | openthaigpt: 5 | pathumma: 5 | kbtg: 5
Q  2 | typhoon: 7 | openthaigpt: 7 | pathumma: 7 | kbtg: 7
  [typhoon] Error: 502 Server Error: Bad Gateway for url: http://thaillm.or.th/api/typhoon/v1/chat/completions, retrying in 1s...
Q  3 | typhoon: 2 | openthaigpt: 2 | pathumma: 2 | kbtg: 2
Q  4 | typhoon: 6 | openthaigpt: 6 | pathumma: 6 | kbtg: 6
Q  5 | typhoon: 6 | openthaigpt: 6 | pathumma: 6 | kbtg: 6
  [typhoon] Error: 502 Server Error: Bad Gateway for url: http://thaillm.or.th/api/typhoon/v1/chat/completions, retrying in 1s...
Q  6 | typhoon: 8 | openthaigpt: 8 | pathumma: 8 | kbtg: 8
  [kbtg] Error: 502 Server Error: Bad Gateway for url: http://thaillm.or.th/api/kbtg/v1/chat/completions, retrying in 1s...
Q  7 | typhoon: 1 | openthaigpt: 1 | pathumma: 1 | kbtg: 1
Q  8 | typhoon: 4 |

In [ ]:
from collections import Counter

def run_voting_pipeline(questions, retrieve_fn, models=["typhoon", "openthaigpt", "pathumma", "thalle"], n=N_QUESTIONS):
    """Run retrieval + Multiple LLMs + Majority Voting."""
    results = []

    for i, q in enumerate(questions[:n]):
        retrieved_chunks = retrieve_fn(q["question"])
        prompt = build_rag_prompt(q["question"], q["choices"], retrieved_chunks)

        model_answers = {}
        for model_name in models:
            raw = ask_llm([
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": prompt},
            ], model=model_name)

            pred = parse_answer(raw)
            model_answers[model_name] = pred
            time.sleep(0.2)

        # Voting logic
        valid_votes = [v for v in model_answers.values() if v is not None]
        if valid_votes:
            final_vote = Counter(valid_votes).most_common(1)[0][0]
        else:
            final_vote = 1 # Default fallback

        results.append({
            "id": q["id"],
            "votes": model_answers,
            "final_answer": final_vote
        })
        print(f"Q{q['id']:>3}: Votes={model_answers} -> Winner={final_vote}")

    return results

# To execute:
# voting_results = run_voting_pipeline(questions, hybrid_retrieve_chunks, n=5)

### 3.3 Compare All Three Methods

In [ ]:
# Define the list of models we have results for
models_to_show = ['typhoon', 'openthaigpt', 'pathumma', 'kbtg']

print(f"{'QID':>4} | {'Method':>7} | {'TYPHOON':>7} | {'OPENGPT':>7} | {'PATHUM':>7} | {'KBTG':>7}")
print("-" * 70)

for q in questions[:10]: # Showing first 10 for readability
    qid = q['id']

    # Row for Dense
    d_row = [dense_preds_multi[m].get(qid, '-') for m in models_to_show]
    print(f"Q{qid:>3} | {'Dense':>7} | {d_row[0]:>7} | {d_row[1]:>7} | {d_row[2]:>7} | {d_row[3]:>7}")

    # Row for BM25
    b_row = [bm25_preds[m].get(qid, '-') for m in models_to_show]
    print(f"Q{qid:>3} | {'BM25':>7} | {b_row[0]:>7} | {b_row[1]:>7} | {b_row[2]:>7} | {b_row[3]:>7}")

    # Row for Hybrid
    if 'hybrid_preds_multi' in globals():
        h_row = [hybrid_preds_multi[m].get(qid, '-') for m in models_to_show]
        print(f"Q{qid:>3} | {'Hybrid':>7} | {h_row[0]:>7} | {h_row[1]:>7} | {h_row[2]:>7} | {h_row[3]:>7}")
    else:
        print(f"Q{qid:>3} | {'Hybrid':>7} | {'-':>7} | {'-':>7} | {'-':>7} | {'-':>7}")

    print("-" * 70)

 QID |  Method | TYPHOON | OPENGPT |  PATHUM |    KBTG
----------------------------------------------------------------------
Q  1 |   Dense |       5 |       5 |       5 |       5
Q  1 |    BM25 |       5 |       5 |       5 |       5
Q  1 |  Hybrid |       5 |       5 |       5 |       5
----------------------------------------------------------------------
Q  2 |   Dense |       7 |       7 |       7 |       7
Q  2 |    BM25 |       7 |       7 |       7 |       7
Q  2 |  Hybrid |       7 |       7 |       7 |       7
----------------------------------------------------------------------
Q  3 |   Dense |       2 |       2 |       2 |       2
Q  3 |    BM25 |       2 |       2 |       2 |       2
Q  3 |  Hybrid |       2 |       2 |       2 |       2
----------------------------------------------------------------------
Q  4 |   Dense |       6 |       6 |       6 |       6
Q  4 |    BM25 |       6 |       6 |       6 |       6
Q  4 |  Hybrid |       6 |       6 |       6 |       6
-

In [ ]:
def get_super_vote(qid, dense_dict, bm25_dict, hybrid_dict, rerank_dict, models=['typhoon', 'openthaigpt', 'pathumma', 'kbtg']):
    all_votes = []
    for model in models:
        v = [dense_dict.get(model, {}).get(qid),
             bm25_dict.get(model, {}).get(qid),
             hybrid_dict.get(model, {}).get(qid),
             rerank_dict.get(model, {}).get(qid)]
        all_votes.extend([ans for ans in v if ans is not None])

    if not all_votes: return 1
    return Counter(all_votes).most_common(1)[0][0]

# Check if rerank_preds_multi is defined, if not, initialize it as an empty dictionary
if 'rerank_preds_multi' not in globals():
    rerank_preds_multi = {}

# Finalizing results
voted_preds = {q['id']: get_super_vote(q['id'], dense_preds_multi, bm25_preds, hybrid_preds_multi, rerank_preds_multi) for q in questions[:N_QUESTIONS]}

In [ ]:
voted_preds

{1: 5,
 2: 7,
 3: 2,
 4: 6,
 5: 6,
 6: 8,
 7: 1,
 8: 4,
 9: 4,
 10: 2,
 11: 4,
 12: 1,
 13: 6,
 14: 4,
 15: 7,
 16: 1,
 17: 8,
 18: 5,
 19: 2,
 20: 2,
 21: 3,
 22: 6,
 23: 3,
 24: 3,
 25: 5,
 26: 6,
 27: 2,
 28: 7,
 29: 4,
 30: 3,
 31: 4,
 32: 2,
 33: 8,
 34: 5,
 35: 3,
 36: 2,
 37: 8,
 38: 6,
 39: 4,
 40: 8,
 41: 7,
 42: 2,
 43: 4,
 44: 1,
 45: 2,
 46: 1,
 47: 2,
 48: 8,
 49: 6,
 50: 5,
 51: 7,
 52: 4,
 53: 9,
 54: 9,
 55: 9,
 56: 9,
 57: 9,
 58: 9,
 59: 9,
 60: 9,
 61: 9,
 62: 9,
 63: 9,
 64: 5,
 65: 3,
 66: 7,
 67: 6,
 68: 1,
 69: 2,
 70: 8,
 71: 4,
 72: 7,
 73: 6,
 74: 5,
 75: 6,
 76: 1,
 77: 7,
 78: 3,
 79: 7,
 80: 1,
 81: 6,
 82: 9,
 83: 4,
 84: 5,
 85: 1,
 86: 1,
 87: 1,
 88: 3,
 89: 4,
 90: 5,
 91: 2,
 92: 3,
 93: 5,
 94: 4,
 95: 1,
 96: 1,
 97: 7,
 98: 3,
 99: 4,
 100: 3}

### Write Submission

Pick your best method and generate a `submission.csv` for Kaggle.

> Set `N_QUESTIONS = 100` at the top and re-run the notebook to generate a full submission.

In [ ]:
with open("submission_all_model.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["id", "answer"])
    for q in questions:
        qid = q["id"]
        # Use voted prediction if available, else default to 1
        writer.writerow([qid, voted_preds.get(qid, 1)])

print(f"submission_all_model.csv written successfully! ({len(questions)} rows)")

submission_all_model.csv written successfully! (100 rows)


In [ ]:
from google.colab import files
files.download('submission_all_model.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

---
## Section 4: What's Next?

This baseline uses a simple setup. Here are ideas to improve your score:

- **Better embeddings** — try other, stronger multilingual  embedding.
- **Smarter chunking** — split by structure or other methods or add useful information to each chunk
- **Chunk size tuning** — experiment with  256, 512, 1024 or something else character chunks
- **Different ThaiLLMs** — try `openthaigpt`, `kbtg`, `pathumma`.
- **Prompt engineering** — adjust the system prompt, add few-shot examples, or change the output format
- **Reranking** — use a cross-encoder or specialized reranker to re-score the top-k chunks before sending to the

**Feel free to implement your own RAG.**